In [25]:
from google.colab import drive
drive.mount('/content/drive')

#!git clone https://github.com/TomLi515/Campus_Life_Coach.git

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# Inference initial definitions
# Our sample rate was 50 Hz so 50 samples per second
# I chose 2s window length so we will have 100 samples per guess
# We will do a 1s hop size so we will do 50% overlap
# [a,b,c,d] first sample is ab second is 1 sec ahead and is bc the cd
#model = models.py
#weights = fusion_classifier.pth,phone_classifier.pth,watch_classifier.pth
# Length in seconds
WINDOW_LENGTH = 2
HOP_LENGTH = 1
# What we sampled
SAMPLE_RATE = 50
# Actual converted length
WINDOW_SAMPLES = int(WINDOW_LENGTH * SAMPLE_RATE)
HOP_SAMPLES = int(HOP_LENGTH * SAMPLE_RATE)
# This is from finetuning Campus_life_coach/finetune/scripts/finetune_all_three_models.py
activity_map = {0 : 'walk', 1 : 'run', 2 : 'sit', 3 : 'stand', 4 : 'lie'}


In [29]:
import sys
sys.modules.pop('finetune.models', None)
sys.modules.pop('finetune', None)


<module 'finetune' (namespace) from ['/content/Campus_Life_Coach/finetune', '/content/Campus_Life_Coach/finetune']>

In [31]:
import torch
import sys
#sys.path.append("Campus_Life_Coach/finetune/src/finetune")
#sys.path.append("/content/Campus_Life_Coach/finetune/src")
#sys.path.append("/content/Campus_Life_Coach/pretrain/src")
sys.path.append('/content/drive/MyDrive/Colab Notebooks/sliding_window/inference/Campus_Life_Coach-main/finetune/src')
sys.path.append('/content/drive/MyDrive/Colab Notebooks/sliding_window/inference/Campus_Life_Coach-main/pretrain/src/pretrain/models')
#sys.modules.pop('finetune.models',None)
#sys.path.append("Campus_Life_Coach") # For pretrain there are pre train calls in finetune
from finetune.models import SingleStreamClassifier, FusionClassifier
# Import in base models
dummy_backbone = torch.nn.Identity()
phone_model = SingleStreamClassifier(pretrained_backbone=dummy_backbone, num_classes =5)
watch_model = SingleStreamClassifier(pretrained_backbone=dummy_backbone, num_classes =5)
fuision_model = FusionClassifier(pretrained_phone_backbone=dummy_backbone, pretrained_watch_backbone=dummy_backbone,num_classes =5)

# Import in weights to model
fuision_model.load_state_dict(torch.load("Campus_Life_Coach/finetune/models/dashboard_models/fusion_classifier.pth",map_location="cpu"))
phone_model.load_state_dict(torch.load("Campus_Life_Coach/finetune/models/dashboard_models/phone_only_classifier.pth",map_location="cpu"))
watch_model.load_state_dict(torch.load("Campus_Life_Coach/finetune/models/dashboard_models/watch_only_classifier.pth",map_location="cpu"))

# Putting it into inference mode we're telling it to stop updating everything weights etc
fuision_model.eval()
phone_model.eval()
watch_model.eval()


ModuleNotFoundError: No module named 'pretrain.models'

In [ ]:
def inference(phone_model,fusion_model,watch_model,phone_window, watch_window):
  # Keep unsqueeze to maintain shape
  phone_tensor = torch.tensor(phone_window, dtype=torch.float32).unsqueeze(0)
  watch_tensor = torch.tensor(watch_window, dtype=torch.float32).unsqueeze(0)
# Pass tensor into model and take softmax of logits (log it = model(tensor))
  phone_probs = torch.softmax(phone_model(phone_tensor),dim=1)
  watch_probs = torch.softmax(watch_model(watch_tensor),dim=1)

# Combine them so we get the both the values
  fusion_input = torch.cat([phone_probs, watch_probs],dim=1)
# Now we are finding the probability of both of them
  fusion_prob = torch.softmax(fuision_model(fusion_input),dim=1)
  return fusion_prob



In [ ]:
# Splitting samples and doing inference on each group

def update_samples(phone_samples, watch_samples, phone_sample, watch_sample):
  phone_samples.append(phone_sample)
  watch_samples.append(watch_sample)
  if (len(phone_samples) >= WINDOW_SAMPLES and len(watch_samples) >= WINDOW_SAMPLES):
    phone_window = phone_samples
    watch_window = watch_samples
# Perform inference on group
#Normilazation is done inside classifier
    probabilities = inference(phone_window, watch_window)
    phone_samples = phone_samples[HOP_SAMPLES:]
    watch_samples = watch_samples[HOP_SAMPLES:]

    return probabilities, phone_samples, watch_samples
  return None, phone_samples,watch_samples




In [ ]:
activity_map = {0 : 'walk', 1 : 'run', 2 : 'sit', 3 : 'stand', 4 : 'lie'}

def most_frequent(list,activity_map):
  walk_count = 0
  run_count = 0
  sit_count = 0
  stand_count = 0
  lie_count = 0
  for i in range(len(list)):
    ele = list[i]
    if ele == 'walk':
      walk_count++
    elif ele == 'sit':
      sit_count++
    elif ele == 'stand':
      stand_count++
    elif ele == 'lie':
      lie_count++
    elif ele == 'run':
      run_count++
  count_list = [walk_count,run_count,sit_count,stand_count,lie_count]
  return activity_map[count_list.argmax().item()]


In [ ]:
import requests
import time
import pandas as pd
# Get current not old data
def get_phyphonx_data(url,last_timestamp):
  response = requests.get(url,timeout=0.2)
  data = response.json()

  # convert to dataframe so we can filter out old times
  df = pd.DataFrame({"Time": data["buffer"]["time"]["values"],
                     "Acc_X": data["buffer"]["acc_x"]["values"],
                     "Acc_Y": data["buffer"]["acc_y"]["values"],
                     "Acc_Z": data["buffer"]["acc_z"]["values"],
                     "Gyro_X": data["buffer"]["gyro_x"]["values"],
                     "Gyro_Y": data["buffer"]["gryo_y"]["values"],
                     "Gyro_Z": data["buffer"]["gyro_z"]["values"]})
  # filter out old times
  if last_timestamp is not None:
    df = df[df["Time"] > last_timestamp]
  # find newest time available
  if len(df) > 0:
    last_timestamp = df["Time"].iloc[-1]

  sample = []
  for  _, row in df.iterrows():
    sample.append([float(row.Acc_X),
                   float(row.Acc_Y),
                   float(row.Acc_Z),
                   float(row.Gyro_X),
                   float(row.Gyro_Y),
                   float(row.Gyro_Z)])

  return sample, last_timestamp





In [ ]:
# Set up the PHYPHOX data

PHYPHOX_URL = "http://10.0.0.227"
print("Getting data from live IMU >:)")
last_time = None
# For buffer and inference steps
program_run = True
phone_samples = []
watch_samples = []
predict_probs = []

# This is for Hysteresis
runner_up_label = None
current_label = None
# counter in case we pick a very bad runner up label
stale_counter = 0
while(program_run):
  # Probability Smoothing we are going to use last 3 windows to start
  # we can do probability smooth(rolling average aka average last n probs then take argmax) or Majority vote keep most common label
  # I tested only with phone so we might wanna test with apple watch to test which smoothing technique is the best

  #Rolling Average vs Majority wins
  if (len(predict_probs) == 3):
    # Rolling Average ===============================================
    smoothed_probs = sum(predict_probs)/3
    predicted_idx_final = smoothed_probs.argmax().item()
    official_predicted_label = activity_map[predicted_idx_final]
    # Majority wins===================================================
    #label_vec = []
    #for i in range(len(predict_probs)):
    #  specific_prob = predict_probs[i]
    #  specific_pred_idx = specific_prob.argmax().item()
    #  label_vec.append(activity_map[specific_pred_idx])
    #official_predicted_label = most_frequent(label_vec,activity_map)
    #==================================================================

    # Performing Hysteresis
    if(current_label == None):
      curent_label = official_predicted_label
    else:
      # if it has a current label it has to prove it isn't a mistake
      # Case 0: Predicting normally continue on
      if (current_label == official_predicted_label):
        continue
      # Official label does not equal current label

      # Case 1: we have perdicted this label and the next prediction also predicted this label we are sure
      elif (runner_up_label == official_predicted_label):
        current_label = runner_up_label
        runner_up_label = None
        # Case 2: have not predicted this label in a row yet so we are unsure
      elif (runner_up_label == None):
        runner_up_label = official_predicted_label
        # Case 3: current label is defined we also habe runner up but new official prediction is different from both
      elif(runner_up_label != official_predicted_label):
        stale_counter++
        continue
    # If we see runner_up_label not fitting with predictions enough we will reset it
    if (stale_counter >= 2):
      runner_up_label = None

    print(current_label)



    # get rid of oldest peice of data
    predict_probs.pop(0)
  while(len(predict_probs)< 3):
    phone_sample, last_time_return = get_phyphonx_data(PHYPHOX_URL, last_time)
    # Ease of testing
    last_time = last_time_return

    watch_sample = phone_sample


    prob,p_sample,w_sample = update_samples(phone_samples,watch_samples,phone_sample,watch_sample)
    # If not none we know we can make a prediction!
    if prob != None:
  # We should have a probability distribution now of both modalities combined!
      predicted_idx = prob.argmax().item()
      predicted_label = activity_map[predicted_idx]
      print(f"not actual label just tests {predicted_label}")
      predict_probs.append(prob)
    else:
      continue


In [ ]:
# Live reading in data